# Run the CS182 adaptation sweep

In Colab, choose **Runtime → Change runtime type → GPU**. Choose **Run all** and authorize the Drive mount. The notebook clones the project into `MyDrive/cs182-sweep/repo` on first use and runs `git pull --ff-only` on later sessions. Data, models, and runs live alongside the checkout in `MyDrive/cs182-sweep/`. The default FP16 experiment runs all 12,306 evaluations and nine adapters, in ICL → LoRA → base order. A high-memory GPU is preferable for the 256-example context. The first run asks to mount Drive and, if needed, for a Hugging Face read token. Your account needs [GPQA dataset access](https://huggingface.co/datasets/Idavidrein/gpqa).

Colab stages the code, prepared data, selected model, and saved run under `/content` before use. During the sweep it snapshots completed evaluations and adapters to Drive about once a minute and again on exit. A sudden VM loss can require repeating work since the last successful snapshot. After a Colab disconnect, run all cells again with the same settings. An interrupted training job starts that adapter over. Set PROFILE to smoke for a six-evaluation practice run.

In [ ]:
PROJECT_DIR = None  # Optional existing checkout; overrides automatic clone and pull.
REPO_URL = "https://github.com/KG2468/ICLvsLoRA.git"
PROFILE = "full"  # "full" or "smoke"
MAX_WORKERS = 2  # Maximum concurrent model replicas on the selected GPU. Set 1 for serial.
PERSIST_TO_DRIVE = True  # Colab only

import os, platform, shutil, subprocess, sys, tempfile
from pathlib import Path
from importlib.util import find_spec

if PROFILE not in {"full", "smoke"}:
    raise ValueError("Choose full or smoke")
COLAB = find_spec("google") is not None and find_spec("google.colab") is not None
if sys.platform == "darwin" and platform.machine() == "arm64":
    DEVICE, MLX = "metal", "mlx==0.32.4"
elif sys.platform == "linux" and shutil.which("nvidia-smi"):
    details = subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version,compute_cap",
                                       "--format=csv,noheader"], text=True).splitlines()[0]
    name, driver, capability = [part.strip() for part in details.split(",")]
    if float(capability) < 7.5 or tuple(map(int, driver.split("."))) < (550, 54, 14):
        raise RuntimeError(f"Unsupported NVIDIA GPU or driver: {details}")
    DEVICE = "cuda"
    MLX = "mlx[cuda13]==0.32.4" if int(driver.split(".")[0]) >= 580 else "mlx[cuda12]==0.32.4"
    print(details)
else:
    raise RuntimeError("Select a Colab GPU runtime or use an Apple Silicon Jupyter kernel.")
DRIVE_STORE = None
if COLAB and PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_STORE = Path("/content/drive/MyDrive/cs182-sweep")
    DRIVE_STORE.mkdir(parents=True, exist_ok=True)
if COLAB:
    STORE = Path("/content/cs182-sweep-local" if DRIVE_STORE else "/content/cs182-sweep")
else:
    STORE = Path.cwd()
STORE.mkdir(parents=True, exist_ok=True)
WORK = Path(tempfile.gettempdir()) / "cs182-notebook"
WORK.mkdir(exist_ok=True)
print("Device", DEVICE, " | Storage", STORE)

## Locate the project checkout and prepare Python 3.12


In [ ]:
if COLAB and PROJECT_DIR is None:
    checkout = (DRIVE_STORE or STORE) / "repo"
    if (checkout / ".git").exists():
        subprocess.run(["git", "-C", str(checkout), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(checkout)], check=True)
    candidates = [checkout]
elif PROJECT_DIR is None:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
else:
    candidates = [Path(PROJECT_DIR).expanduser().resolve()]

ROOT = next((path for path in candidates
             if (path / "src/final_project/__main__.py").is_file()
             and (path / "configs" / f"{PROFILE}.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Set PROJECT_DIR in the first cell to the checkout containing src/ and configs/.")
if COLAB:
    # Copy the runner, configs and storage helper, including checkout edits.
    checkout = ROOT
    ROOT = Path(tempfile.mkdtemp(prefix="source-", dir=STORE))
    for folder in ("src", "configs"):
        shutil.copytree(checkout / folder, ROOT / folder, ignore=shutil.ignore_patterns("__pycache__"))
    shutil.copy2(checkout / "scripts/colab_storage.py", ROOT / "colab_storage.py")
    import runpy
    storage = runpy.run_path(str(ROOT / "colab_storage.py"))
else:
    STORE = ROOT
print("Runtime source:", ROOT, "| Experiment storage:", STORE)


In [ ]:
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
UV = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
VENV = WORK / ("venv-" + DEVICE)
subprocess.run([UV, "python", "install", "3.12.11"], check=True)
if not (VENV / "bin/python").exists():
    subprocess.run([UV, "venv", "--python", "3.12.11", str(VENV)], check=True)
PY = str(VENV / "bin/python")
subprocess.run([UV, "pip", "install", "--python", PY, MLX, "mlx-lm[train]==0.32.2",
                "datasets==4.8.5", "transformers==5.23.0", "huggingface-hub==1.33.0",
                "matplotlib==3.11.2"], check=True)
env = {**os.environ, "PYTHONPATH": str(ROOT / "src"), "PYTHONUNBUFFERED": "1"}
def run_python(*args, **kwargs):
    return subprocess.run([PY, *args], env=env, cwd=STORE, check=True, **kwargs)
run_python("-c", "import mlx.core as mx; mx.set_default_device(mx.gpu); "
                 "print(mx.device_info()); print('GPU check:', (mx.ones((2,2)) @ mx.ones((2,2))).tolist())")
if DEVICE == "cuda":
    import ctypes.util
    if not ctypes.util.find_library("seccomp"):
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "libseccomp2"], check=True)
    run_python("-c", "from final_project.scoring import execute; "
               "r=execute('import math\\ndef f(): return math.factorial(4)', "
               "'def check(f): assert f()==24', 'f', 10); "
               "assert r['passed'], r; print('Linux scoring check passed')")
    # Exercise the isolation boundary before giving the scorer model output.
    probe = """import os, socket
def f():
    blocked = 0
    for action in (lambda: open('/etc/passwd'), lambda: socket.socket(), lambda: os.fork()):
        try:
            action()
        except OSError:
            blocked += 1
    return blocked
"""
    run_python("-c", "from final_project.scoring import execute; "
               + "r=execute(" + repr(probe) + ", 'def check(f): assert f()==3', 'f', 10); "
               + "assert r['passed'], r")

## Data access and model setup

The notebook checks for a cached GPQA dataset and a saved Hugging Face login. If login is needed, it reads the Colab secret named **HF_TOKEN** or prompts through a hidden input. Dataset approval is managed by Hugging Face.

In [ ]:
from getpass import getpass
import tomllib

CONFIG = WORK / ("run-" + PROFILE + ".toml")
text = (ROOT / "configs" / (PROFILE + ".toml")).read_text()
import re
text = re.sub(r'(?m)^name = "([^"]+)"', lambda m: f'name = "{m[1]}-notebook-{DEVICE}"', text)
if DEVICE == "cuda":
    text = text.replace('sandbox = "macos"', 'sandbox = "linux-chroot"')
if type(MAX_WORKERS) is not int or MAX_WORKERS < 1:
    raise ValueError("MAX_WORKERS must be a positive integer")
text = re.sub(r'(?m)^max_workers = .*$', f'max_workers = {MAX_WORKERS}', text)
CONFIG.write_text(text)
cfg = tomllib.loads(CONFIG.read_text())
exp = cfg["experiment"]
data_dir = Path(exp["data_dir"])
model_dir = Path(cfg["model"]["directory"]) / cfg["model"]["precision"]
run_dir = Path(exp["output_dir"]) / exp["name"]
if DRIVE_STORE:
    for directory in (data_dir, model_dir, run_dir):
        storage["copy_tree_once"](DRIVE_STORE / directory, STORE / directory)

if PROFILE == "full" and not (STORE / data_dir / "gpqa.jsonl").is_file():
    auth_check = "from huggingface_hub import HfApi; HfApi().whoami()"
    result = subprocess.run([PY, "-c", auth_check], env=env, capture_output=True)
    if result.returncode:
        token = None
        if COLAB:
            from google.colab import userdata
            try:
                token = userdata.get("HF_TOKEN")
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
        if not token:
            token = getpass("Hugging Face read token for GPQA: ").strip()
        try:
            run_python("-c", "import sys; from huggingface_hub import login; "
                       "login(token=sys.stdin.read(), add_to_git_credential=False)",
                       input=token, text=True)
        finally:
            del token

def project(command):
    run_python("-m", "final_project", command, "--config", str(CONFIG))

project("prepare")
if DRIVE_STORE:
    storage["save_data"](STORE / data_dir, DRIVE_STORE / data_dir)

if COLAB and not (STORE / model_dir).exists():
    # Publish a converted model only after the model command succeeds.
    with tempfile.TemporaryDirectory(dir=STORE) as staging:
        subprocess.run([PY, "-m", "final_project", "model", "--config", str(CONFIG)],
                       cwd=staging, env=env, check=True)
        (STORE / model_dir).parent.mkdir(parents=True, exist_ok=True)
        (Path(staging) / model_dir).rename(STORE / model_dir)
else:
    project("model")
if DRIVE_STORE:
    storage["copy_tree_once"](STORE / model_dir, DRIVE_STORE / model_dir)
project("plan")


## Run or resume the sweep

Training, scoring, and result writes use local storage. With Drive persistence enabled, the notebook backs up new trace records, completed adapters, and reports in the background of the running experiment. Use one active runtime per run directory.


In [ ]:
if DRIVE_STORE:
    backup = storage["RunBackup"](STORE / run_dir, DRIVE_STORE / run_dir)
    backup.run([PY, "-m", "final_project", "run", "--config", str(CONFIG)], env, STORE)
    print("Drive snapshot:", DRIVE_STORE / run_dir)
else:
    project("run")


## Inspect results

In [ ]:
from IPython.display import Markdown, Image, display
name = re.search(r'(?m)^name = "([^"]+)"', CONFIG.read_text())[1]
output = STORE / "runs" / name
display(Markdown((output / "report.md").read_text()))
display(Image(filename=str(output / "accuracy.png")))
print("Scores:", output / "summary.csv")
print("Prompt and output traces:", output / "results.jsonl")